# Evolus versus simulator assignments

Run the three code cells in order. The first opens the dashboard. The second resets the seeded scenario, streams a reply from Evolus, and applies validated assignments before each simulated minute. The third resets the same seed and uses the simulator’s assignment policy, then prints side-by-side results and differences. Keep the dashboard open while the runs execute. An Evolus API key is required for the second cell.


In [ ]:

import subprocess
import time
import webbrowser
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import urlopen

_current = Path.cwd().resolve()
_candidates = [_current / "digital-twin"]

if _current.name.lower() == "digital-twin":
    _candidates.insert(0, _current)

_candidates.extend(parent / "digital-twin" for parent in _current.parents)

project_root = next(
    (path for path in _candidates if path.is_dir()), None
)

if project_root is None:
    raise FileNotFoundError(
        f"Could not find a digital-twin folder from {_current}"
    )

ports = (4173, 5173)


def find_app():
    for port in ports:
        try:
            with urlopen(
                f"http://127.0.0.1:{port}/",
                timeout=2
            ) as response:
                if response.status == 200:
                    return f"http://127.0.0.1:{port}"

        except (HTTPError, URLError, TimeoutError, OSError):
            continue

    return None


app_url = find_app()

if app_url is None:
    print(f"Starting the app from {project_root}...", flush=True)

    server = subprocess.Popen(
        ["cmd.exe", "/c", "npm.cmd run dev"],
        cwd=str(project_root)
    )

    deadline = time.monotonic() + 60

    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(
                "The app server exited. Check the npm output."
            )

        app_url = find_app()

        if app_url:
            break

        time.sleep(1)

    if app_url is None:
        raise TimeoutError(
            "The dashboard did not respond within 60 seconds."
        )

print(f"App is ready at {app_url}", flush=True)
webbrowser.open(app_url)
print(
    "The dashboard is opening. "
    "Once it loads, run the next cell to start the simulation."
)


In [ ]:
import getpass
import json
import os
import re
import time
from queue import Empty, Queue
from threading import Thread
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

EVOLUS_AGENT_ID = "ccf70666-7d77-44d9-b803-6b4553b29cbf"
EVOLUS_APPLICATION_ID = "50d9b290-0675-45ec-a49c-885d89e597ec"
TWIN_APIS = [
    "http://127.0.0.1:4173/api/simulation-state",
    "http://127.0.0.1:5173/api/simulation-state",
]

api_key = globals().get("api_key") or os.environ.get("EVOLUS_API_KEY") or getpass.getpass("Evolus API key: ")
if not api_key:
    raise RuntimeError("An Evolus API key is required to ask the assignment agent.")

def read_twin_snapshot(endpoint):
    with urlopen(endpoint, timeout=10) as response:
        return json.load(response)["snapshot"]


def send_twin_command(endpoint, run_id, action, **fields):
    request = Request(
        endpoint,
        data=json.dumps({"action": action, "runId": run_id, **fields}).encode("utf-8"),
        headers={"Content-Type": "application/json"}, method="POST",
    )
    with urlopen(request, timeout=10) as response:
        command = json.load(response)
    deadline = time.monotonic() + 150
    reported_delay = False
    while time.monotonic() < deadline:
        with urlopen(f"{endpoint}?id={command['id']}", timeout=10) as response:
            result = json.load(response)
        if result["status"] == "applied":
            return result
        if result["status"] != "pending":
            raise RuntimeError(f"{action} rejected: {result.get('message')}")
        if not reported_delay and time.monotonic() > deadline - 130:
            print(f"{action} is still queued ({command['id']}); bring the dashboard tab to the foreground so it can process commands.", flush=True)
            reported_delay = True
        time.sleep(0.5)
    raise TimeoutError(f"{action} remained queued for 150 seconds (command {command['id']}). Bring the dashboard tab to the foreground and check the simulation state before retrying.")


def parse_evolus_assignments(agent_result, snapshot):
    """Extract explicit patient-doctor-room choices from the agent response."""
    answer = agent_result.get("answer")
    if answer is None or answer == "":
        raise ValueError("Evolus returned no assignment decision.")
    preview = str(answer)
    print("Evolus decision:", preview[:1500] + ("..." if len(preview) > 1500 else ""), flush=True)
    proposal = answer
    if isinstance(answer, str):
        text = answer.strip()
        fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", text, flags=re.I | re.S)
        if fenced:
            text = fenced.group(1)
        try:
            proposal = json.loads(text)
        except json.JSONDecodeError:
            proposal = None
            decoder = json.JSONDecoder()
            for match in re.finditer(r"[\[{]", text):
                try:
                    candidate, _ = decoder.raw_decode(text[match.start():])
                except json.JSONDecodeError:
                    continue
                if isinstance(candidate, list) or (isinstance(candidate, dict) and "assignments" in candidate):
                    proposal = candidate
                    break

    if isinstance(proposal, dict):
        proposal = proposal.get("assignments")
    if isinstance(proposal, list):
        def resolve(value, records, labels):
            if not isinstance(value, str):
                return value
            value = value.strip().casefold()
            for record in records:
                if any(isinstance(record.get(label), str) and record[label].casefold() == value for label in labels):
                    return record["id"]
            return value

        waiting = snapshot.get("waiting", [])
        doctors = [item for item in snapshot.get("staff", []) if item.get("role") == "doctor"]
        rooms = snapshot.get("rooms", [])
        normalized = []
        for item in proposal:
            if not isinstance(item, dict):
                normalized.append(item)
                continue
            normalized.append({
                "patient_id": resolve(item.get("patient_id", item.get("patientId", item.get("patient"))), waiting, ("id", "name")),
                "doctor_id": resolve(item.get("doctor_id", item.get("doctorId", item.get("doctor"))), doctors, ("id", "name")),
                "room_id": resolve(item.get("room_id", item.get("roomId", item.get("room"))), rooms, ("id", "label")),
                "reason": item.get("reason", ""),
            })
        return normalized

    # Accept a line of prose only when it identifies exactly one of each entity.
    if isinstance(answer, str):
        def mentioned(line, records, labels):
            found = []
            for record in records:
                if any(isinstance(record.get(label), str) and re.search(r"(?<!\w)" + re.escape(record[label]) + r"(?!\w)", line, flags=re.I) for label in labels):
                    found.append(record["id"])
            return set(found)

        choices = []
        waiting = snapshot.get("waiting", [])
        doctors = [item for item in snapshot.get("staff", []) if item.get("role") == "doctor"]
        rooms = snapshot.get("rooms", [])
        for line in answer.splitlines():
            if re.search(r"\b(?:do not|don't|avoid|cannot|can't|should not|shouldn't|no)\b", line, flags=re.I):
                continue
            if not re.search(r"\b(?:assign(?:ed|ment)?|place(?:d)?|pair(?:ing|ed)?|move(?:d)?)\b|->|→", line, flags=re.I):
                continue
            patients = mentioned(line, waiting, ("id", "name"))
            doctors_found = mentioned(line, doctors, ("id", "name"))
            rooms_found = mentioned(line, rooms, ("id", "label"))
            if len(patients) == len(doctors_found) == len(rooms_found) == 1:
                choices.append({
                    "patient_id": next(iter(patients)),
                    "doctor_id": next(iter(doctors_found)),
                    "room_id": next(iter(rooms_found)),
                    "reason": line.strip(),
                })
        if choices:
            return choices
        if re.search(r"\b(no|none|zero)\s+(?:safe |feasible |valid )?assignments?\b", answer, flags=re.I):
            return []
    raise ValueError("Evolus returned a summary instead of patient-doctor-room assignments. The simulator remains paused. Check whether the agent endpoint supports the requested JSON response schema.")


def apply_evolus_assignments(snapshot, active_api):
    run_id = snapshot["runId"]
    if not snapshot.get("waiting"):
        print("No waiting patients; advancing without an assignment request.", flush=True)
        return
    waiting = snapshot.get("waiting", [])
    staff = snapshot.get("staff", [])
    rooms = snapshot.get("rooms", [])
    resources = snapshot.get("resources", [])
    available_doctors = [
        member for member in staff
        if member.get("role") == "doctor" and member.get("status") == "available"
    ]
    available_nurses = [
        member for member in staff
        if member.get("role") == "nurse" and member.get("status") == "available"
    ]

    # Publish current legal choices as simulation-derived data; Evolus still selects the assignments.
    resource_by_type = {resource["type"]: resource for resource in resources}
    assignment_options = []
    for patient in waiting:
        doctor_ids = [
            doctor["id"] for doctor in available_doctors
            if patient.get("specialty") in (None, "General") or
            doctor.get("specialty") == patient.get("specialty")
        ]
        room_ids = [
            room["id"] for room in rooms
            if room.get("type") == patient.get("room") and (
                room.get("status") == "ready" or
                (room.get("status") == "reserved" and room.get("reservedForPatientId") == patient.get("id"))
            )
        ]
        resources_ready = all(
            (resource := resource_by_type.get(resource_type)) is not None and
            resource.get("status") != "offline" and
            resource.get("used", 0) < resource.get("capacity", 0)
            for resource_type in patient.get("resources", [])
        )
        if doctor_ids and room_ids and resources_ready:
            assignment_options.append({"patient_id": patient["id"], "doctor_ids": doctor_ids, "room_ids": room_ids})
    if not available_nurses or not assignment_options:
        print("No feasible assignment at this minute; advancing to the next state.", flush=True)
        return

    # Send only the simulator's published synthetic state; the agent supplies its own instructions.
    simulation_state = {
        key: snapshot.get(key)
        for key in (
            "runId", "simTime", "simMinute", "stateVersion", "waiting", "treating",
            "inbound", "rooms", "staff", "resources", "metrics", "upcomingIncidents",
        )
    }
    simulation_state["assignmentOptions"] = assignment_options
    simulation_state["maxAssignmentsNow"] = min(
        len(available_nurses), len(assignment_options),
        len({doctor_id for option in assignment_options for doctor_id in option["doctor_ids"]}),
        len({room_id for option in assignment_options for room_id in option["room_ids"]}),
    )
    simulation_data = json.dumps(simulation_state, ensure_ascii=False)

    # The configured Evolus agent decides from the simulation data alone.
    agent_request = Request(
        "https://api.evolus.ai/v1/chat/completions",
        data=json.dumps({
            "model": f"{EVOLUS_APPLICATION_ID}:{EVOLUS_AGENT_ID}",
            "messages": [{"role": "user", "content": simulation_data}],
            "stream": True,
            "max_tokens": 400,
        }).encode("utf-8"),
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
            "Accept": "text/event-stream",
        },
        method="POST",
    )
    print(f"Streaming {len(simulation_data.encode('utf-8')):,} bytes of simulation data to Evolus...", flush=True)
    request_started = time.monotonic()
    responses = Queue()

    def ask_evolus():
        fragments = []
        tool_fragments = {}
        event_count = 0
        last_event_keys = []
        last_choice_keys = []
        last_delta_keys = []
        try:
            with urlopen(agent_request, timeout=150) as response:
                responses.put(("connected", None))
                content_type = response.headers.get("Content-Type", "")
                if "text/event-stream" not in content_type:
                    data = json.load(response)
                    message = data.get("choices", [{}])[0].get("message", {})
                    calls = message.get("tool_calls", [])
                    answer = calls[0]["function"]["arguments"] if calls else message.get("content")
                    responses.put(("result", {"answer": answer}))
                    return
                for raw_line in response:
                    line = raw_line.decode("utf-8", errors="replace").strip()
                    if not line.startswith("data:"):
                        continue
                    data = line[5:].strip()
                    if data == "[DONE]":
                        break
                    chunk = json.loads(data)
                    event_count += 1
                    last_event_keys = list(chunk) if isinstance(chunk, dict) else [type(chunk).__name__]
                    if "error" in chunk:
                        raise RuntimeError(f"Evolus stream error: {chunk['error']}")
                    finished = False
                    for choice in chunk.get("choices", []):
                        last_choice_keys = list(choice)
                        delta = choice.get("delta") or choice.get("message") or {}
                        last_delta_keys = list(delta) if isinstance(delta, dict) else [type(delta).__name__]
                        if not isinstance(delta, dict):
                            continue
                        for call in delta.get("tool_calls", []):
                            function = call.get("function", {})
                            if function.get("name") not in (None, "submit_assignments"):
                                raise RuntimeError("Evolus called an unexpected function")
                            index = call.get("index", 0)
                            piece = function.get("arguments", "")
                            if piece:
                                if not tool_fragments:
                                    responses.put(("first_token", None))
                                tool_fragments[index] = tool_fragments.get(index, "") + piece
                                try:
                                    decision = json.loads(tool_fragments[index])
                                except json.JSONDecodeError:
                                    pass
                                else:
                                    if isinstance(decision, dict) and isinstance(decision.get("assignments"), list):
                                        responses.put(("result", {"answer": tool_fragments[index]}))
                                        return
                        content = delta.get("content")
                        if isinstance(content, str) and content:
                            if not fragments and not tool_fragments:
                                responses.put(("first_token", None))
                            fragments.append(content)
                            answer_text = "".join(fragments).strip()
                            if answer_text.startswith(("{", "[")):
                                try:
                                    decision = json.loads(answer_text)
                                except json.JSONDecodeError:
                                    pass
                                else:
                                    if isinstance(decision, list) or (isinstance(decision, dict) and isinstance(decision.get("assignments"), list)):
                                        responses.put(("result", {"answer": answer_text}))
                                        return
                            elif not answer_text.startswith("```") and len(answer_text) >= 40:
                                raise RuntimeError(
                                    "Evolus began a narrative reply instead of assignment JSON: "
                                    + answer_text[:200]
                                )
                        if choice.get("finish_reason") is not None:
                            finished = True
                    if finished:
                        break
                answer = next(iter(tool_fragments.values()), "".join(fragments))
                responses.put(("result", {"answer": answer}))
        except Exception as error:
            partial = next(iter(tool_fragments.values()), "".join(fragments))
            if partial and not isinstance(error, RuntimeError):
                responses.put(("error", RuntimeError(
                    f"Evolus stream stopped after {len(partial)} reply characters: {error}. "
                    f"Partial reply: {partial[:500]}"
                )))
            elif isinstance(error, TimeoutError):
                responses.put(("error", RuntimeError(
                    f"Evolus sent no assignment reply before the 150-second read timeout. "
                    f"Received {event_count} stream events; last event keys: {last_event_keys}; "
                    f"choice keys: {last_choice_keys}; delta/message keys: {last_delta_keys}. "
                    "If only a role was emitted, the configured Evolus model did not generate an answer."
                )))
            else:
                responses.put(("error", error))

    Thread(target=ask_evolus, daemon=True).start()
    try:
        while True:
            remaining = 180 - (time.monotonic() - request_started)
            if remaining <= 0:
                raise TimeoutError("Evolus did not finish within 180 seconds")
            try:
                event, payload = responses.get(timeout=min(5, remaining))
            except Empty:
                print(f"Still waiting for Evolus ({time.monotonic() - request_started:.0f}s)...", flush=True)
                continue
            if event == "connected":
                print("Evolus stream connected.", flush=True)
            elif event == "first_token":
                print("Evolus started replying.", flush=True)
            elif event == "error":
                raise payload
            elif event == "result":
                agent_result = payload
                break
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Evolus chat request failed ({error.code}): {detail[:1000]}") from error
    except (URLError, TimeoutError) as error:
        raise RuntimeError(f"Evolus chat request timed out or failed: {error}") from error
    print(f"Evolus responded in {time.monotonic() - request_started:.1f}s.", flush=True)

    assignments = parse_evolus_assignments(agent_result, snapshot)
    if len(assignments) > 20:
        raise ValueError("Evolus proposed more than the 20-pairing API limit. Nothing was sent.")
    # Match the simulator policy's severity and waiting-time order when choices compete.
    waiting_priority = {patient["id"]: (patient["severity"], -patient["waitMinutes"])
                        for patient in waiting}
    assignments.sort(key=lambda item: waiting_priority.get(
        item.get("patient_id") if isinstance(item, dict) else None,
        (99, 0),
    ))

    waiting_by_id = {patient["id"]: patient for patient in waiting}
    doctors_by_id = {doctor["id"]: doctor for doctor in available_doctors}
    rooms_by_id = {room["id"]: room for room in rooms}
    used_patients = set()
    used_doctors = set()
    used_rooms = set()
    resource_use = {resource["type"]: resource.get("used", 0) for resource in resources}
    resource_by_type = {resource["type"]: resource for resource in resources}
    validated_pairings = []
    skipped_pairings = []

    for index, assignment in enumerate(assignments, start=1):
        if not isinstance(assignment, dict):
            skipped_pairings.append(f"Assignment {index}: response item is not an object")
            continue
        patient_id = assignment.get("patient_id")
        doctor_id = assignment.get("doctor_id")
        room_id = assignment.get("room_id")
        if not all(isinstance(value, str) and value for value in (patient_id, doctor_id, room_id)):
            skipped_pairings.append(f"Assignment {index}: missing patient, doctor, or room ID")
            continue
        if patient_id in used_patients or doctor_id in used_doctors or room_id in used_rooms:
            skipped_pairings.append(f"Assignment {index}: reuses a selected patient, doctor, or room")
            continue
        patient = waiting_by_id.get(patient_id)
        doctor = doctors_by_id.get(doctor_id)
        room = rooms_by_id.get(room_id)
        if not patient or not doctor or not room:
            skipped_pairings.append(f"Assignment {index}: patient is not waiting, doctor is unavailable, or room is unknown")
            continue
        if room.get("status") != "ready" and not (
            room.get("status") == "reserved" and room.get("reservedForPatientId") == patient_id
        ):
            skipped_pairings.append(f"Assignment {index}: room is not ready or reserved for this patient")
            continue
        if room.get("type") != patient.get("room"):
            skipped_pairings.append(f"Assignment {index}: room type does not match {patient_id}")
            continue
        specialty = patient.get("specialty")
        if specialty and specialty != "General" and doctor.get("specialty") != specialty:
            skipped_pairings.append(f"Assignment {index}: doctor lacks {patient_id}'s required specialty")
            continue
        if len(validated_pairings) >= len(available_nurses):
            skipped_pairings.append(f"Assignment {index}: no available nurse remains")
            continue
        required_types = []
        resource_problem = None
        for resource_type in patient.get("resources", []):
            resource = resource_by_type.get(resource_type)
            if resource is None or resource.get("status") == "offline":
                resource_problem = f"required resource {resource_type} is unavailable"
                break
            if resource_use[resource_type] + required_types.count(resource_type) >= resource.get("capacity", 0):
                resource_problem = f"required resource {resource_type} is at capacity"
                break
            required_types.append(resource_type)
        if resource_problem:
            skipped_pairings.append(f"Assignment {index}: {resource_problem}")
            continue
        for resource_type in required_types:
            resource_use[resource_type] += 1
        used_patients.add(patient_id)
        used_doctors.add(doctor_id)
        used_rooms.add(room_id)
        validated_pairings.append({
            "patient_id": patient_id,
            "doctor_id": doctor_id,
            "room_id": room_id,
            "reason": str(assignment.get("reason", "")),
        })

    for reason in skipped_pairings:
        print(f"Skipped Evolus proposal: {reason}", flush=True)

    if not validated_pairings:
        print("Evolus found no feasible assignments. No command was sent.")
    else:
        print("Validated pairings (submitting automatically):")
        for pairing in validated_pairings:
            patient = waiting_by_id[pairing["patient_id"]]
            doctor = doctors_by_id[pairing["doctor_id"]]
            room = rooms_by_id[pairing["room_id"]]
            print(
                f"- {patient['id']} {patient['name']} -> {doctor['name']} / {room['label']}"
                + (f" ? {pairing['reason']}" if pairing["reason"] else "")
            )
        print(
            f"Based on run {run_id}, state version {snapshot.get('stateVersion')}. "
            "Submitting automatically; the simulator rechecks live constraints."
        )
        control_api = active_api.rsplit("/", 1)[0] + "/simulation-control"
        result = send_twin_command(control_api, run_id, "assign_selected",
            expectedStateVersion=snapshot["stateVersion"], pairings=[
            {key: pairing[key] for key in ("patient_id", "doctor_id", "room_id")}
            for pairing in validated_pairings
        ])
        print(result.get("message", "Assignment request applied."), flush=True)
        if result.get("assignedCount") != len(validated_pairings):
            raise RuntimeError(
                f"The simulator applied {result.get('assignedCount')} of {len(validated_pairings)} proposed assignments. "
                "Simulation is paused; inspect the reasons above."
            )

# Each step advances one simulated minute with the built-in assignment policy disabled.
active_api = None
for candidate in TWIN_APIS:
    try:
        initial = read_twin_snapshot(candidate)
        active_api = candidate
        break
    except (HTTPError, URLError, TimeoutError):
        continue
if active_api is None:
    raise ConnectionError("Start the app and keep its dashboard open before running this cell.")
run_id = initial["runId"]
control_api = active_api.rsplit("/", 1)[0] + "/simulation-control"
agent_comparison_result = None
print("Evolus controller: streaming agent response; keep the dashboard open.", flush=True)
try:
    send_twin_command(control_api, run_id, "pause")
    send_twin_command(control_api, run_id, "reset")
    print("Evolus comparison run: reset to the scenario seed.", flush=True)
    while True:
        snapshot = read_twin_snapshot(active_api)
        if snapshot.get("runId") != run_id:
            raise RuntimeError("The active run changed. Restart this cell for the new run.")
        if snapshot.get("complete"):
            agent_comparison_result = snapshot
            print("\nEvolus run results:", flush=True)
            print(f"  Simulated minutes: {snapshot['simMinute']}", flush=True)
            print(f"  Patients discharged: {snapshot['metrics']['discharged']}", flush=True)
            print(f"  Mean wait: {snapshot['metrics']['meanWait']} minutes", flush=True)
            print(f"  Peak waiting queue: {snapshot['metrics']['peakQueue']}", flush=True)
            print(f"  Deterioration events: {snapshot['metrics']['deteriorationEvents']}", flush=True)
            print("Run the next cell to see the simulator-policy results and differences.", flush=True)
            break
        if snapshot["simMinute"] >= 360:
            raise TimeoutError("Evolus run exceeded 360 simulated minutes; comparison stopped.")
        print(f"Time {snapshot.get('simTime')} | state v{snapshot.get('stateVersion')} | waiting for Evolus", flush=True)
        apply_evolus_assignments(snapshot, active_api)
        send_twin_command(control_api, run_id, "step_agent")
        time.sleep(0.1)
except KeyboardInterrupt:
    print("Evolus controller stopped. Simulation remains paused.", flush=True)
finally:
    # Pause on API, parsing, or validation failures without hiding the original error.
    try:
        send_twin_command(control_api, run_id, "pause")
    except Exception as pause_error:
        print(f"Could not confirm the simulator is paused: {pause_error}", flush=True)


Evolus controller stopped. Simulation remains paused.


In [ ]:
# Run the same seeded scenario using only the simulator's assignment policy.
# Run this cell after the Evolus controller cell completes.
if not isinstance(globals().get("agent_comparison_result"), dict) or not agent_comparison_result.get("complete"):
    raise RuntimeError("Complete the Evolus controller cell before running this comparison.")

comparison_api = active_api
comparison_control = comparison_api.rsplit("/", 1)[0] + "/simulation-control"
agent_result = agent_comparison_result
baseline_comparison_result = None
comparison_run_id = agent_result["runId"]
try:
    send_twin_command(comparison_control, comparison_run_id, "pause")
    send_twin_command(comparison_control, comparison_run_id, "reset")
    baseline_initial = read_twin_snapshot(comparison_api)
    if baseline_initial["seed"] != agent_result["seed"] or baseline_initial["simMinute"] != 0:
        raise RuntimeError("The baseline did not reset to the Evolus run's seed and starting minute.")
    print(f"Simulator policy run started from seed {baseline_initial['seed']}.", flush=True)
    while True:
        baseline = read_twin_snapshot(comparison_api)
        if baseline["runId"] != comparison_run_id:
            raise RuntimeError("The simulation run changed during comparison.")
        if baseline.get("complete"):
            baseline_comparison_result = baseline
            break
        if baseline["simMinute"] >= 360:
            raise TimeoutError("Simulator-policy run exceeded 360 simulated minutes; comparison stopped.")
        # Match the Evolus run's timing: assign waiting patients, then advance one minute.
        send_twin_command(comparison_control, comparison_run_id, "assign_waiting")
        send_twin_command(comparison_control, comparison_run_id, "step_agent")
        if (baseline["simMinute"] + 1) % 10 == 0:
            print(f"Simulator policy: reached minute {baseline['simMinute'] + 1}...", flush=True)
    print("\nOutcome comparison (Difference = Evolus minus simulator):", flush=True)
    print(f"{'Metric':<24} {'Evolus':>12} {'Simulator':>12} {'Difference':>12}")
    print("-" * 64)
    rows = [
        ("Simulated minutes", agent_result["simMinute"], baseline["simMinute"]),
        ("Patients discharged", agent_result["metrics"]["discharged"], baseline["metrics"]["discharged"]),
        ("Mean wait (minutes)", agent_result["metrics"]["meanWait"], baseline["metrics"]["meanWait"]),
        ("Peak waiting queue", agent_result["metrics"]["peakQueue"], baseline["metrics"]["peakQueue"]),
        ("Deterioration events", agent_result["metrics"]["deteriorationEvents"], baseline["metrics"]["deteriorationEvents"]),
    ]
    for label, evolus, simulator in rows:
        difference = evolus - simulator
        print(f"{label:<24} {evolus:>12} {simulator:>12} {difference:>+12g}")
    print("\nFor time, wait, queue, and deterioration, a negative difference favors Evolus.")
    agent_patients = {patient["id"]: patient for patient in agent_result["patients"]}
    baseline_patients = {patient["id"]: patient for patient in baseline["patients"]}
    print("\nPatient wait times (minutes):")
    print(f"{'Patient':<22} {'Evolus':>10} {'Simulator':>10} {'Difference':>12}")
    print("-" * 58)
    for patient_id, patient in agent_patients.items():
        other = baseline_patients.get(patient_id)
        if other is None:
            continue
        evolus_wait = patient["waitMinutes"]
        simulator_wait = other["waitMinutes"]
        print(f"{patient['name']:<22} {evolus_wait:>10} {simulator_wait:>10} {evolus_wait - simulator_wait:>+12}")
    print("Both runs used the same seed and one-minute assignment/advance sequence.", flush=True)
except KeyboardInterrupt:
    print("Comparison stopped. Simulation remains paused.", flush=True)
finally:
    send_twin_command(comparison_control, comparison_run_id, "pause")
